# Where-fi - Object localization using RSSI of Wi-fi signals indoors

### Project for the INTAR lecture at UPV

## Imports and configurations

### Imports of useful libraries

In [ ]:
# Libraries for logging and general utilities
import random
import logging
import time
import math

# Libraries for data manipulation and analysis
import numpy as np
import pandas as pd

# Helpful libraries
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Tuple, Union

# Deep learning libraries
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchmetrics.classification import (AUROC, Accuracy, ConfusionMatrix,
                                         F1Score, MulticlassAUROC, Precision,
                                         Recall)

### Configuration to control model architecture and training

In [ ]:
CONFIG = {
    "seed": 42,
    "batch_size": 64,
    "num_epochs": 50,
    "learning_rate": 0.001,
    "learning_rate_scheduler": "Cosine",
    "model": {
        "name": ""
    },
    "dataset": {
        "root_path": ""
    }
}

## Helper classes
Classes for utility functionalities

### Class for training parameters
Small helper class to organise the parameters for the training

In [ ]:
class TrainingParameters:
    '''
    Class to encapsulate training parameters.
    Contains learning rate, beta values, weight decay, and number of epochs.
    '''

    def __init__(self,
            lr: float,
            lr_min: float,
            lr_scheduler: str,
            beta1: float,
            beta2: float,
            weight_decay: float,
            epochs: int
        ):
        '''
        Args:
            lr (float): Learning rate
            beta1 (float): Beta1 value for optimizer
            beta2 (float): Beta2 value for optimizer
            weight_decay (float): Weight decay for optimizer
            epochs (int): Number of training epochs

        Initializes the training parameters with the given values.
        '''
        self.lr = lr
        self.lr_min = lr_min
        self.lr_scheduler = lr_scheduler
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.epochs = epochs

    def get_learning_rate(self) -> float:
        '''
        Get the learning rate.
        '''
        return self.lr

    def get_learning_rate_min(self) -> float:
        '''
        Get the minimum learning rate.
        '''
        return self.lr_min

    def get_betas(self) -> tuple[float, float]:
        '''
        Get the beta values.
        '''
        return (self.beta1, self.beta2)

    def get_weight_decay(self) -> float:
        '''
        Get the weight decay.
        '''
        return self.weight_decay

    def get_epochs(self) -> int:
        '''
        Get the number of epochs.
        '''
        return self.epochs

    def get_lr_scheduler(self) -> str:
        '''
        Get lr_schedular
        '''
        return self.lr_scheduler

## Dataset

### Dataaugmentation

In [ ]:
class DataAugmentation:
    """
    Class for randomly applying data augmentations of the following list:
        - Amplitude scaling
        - Null masking of some channels of up to 5% of the signal
        - Gaussian noise addition with std up to 0.05
        - Rotation of sensor values
    """
    def __init__(self, cfg: any):
        '''
        Initializes the DataAugmentation with a composition of random
        transformations.
        '''
        # TODO: Add augmentation necessary for this project
        self.transform = T.Compose([
            # GaussianNoise(std_range=(0.01, 0.05), p=cfg.prob_augmentations.gaussian_noise),
            # RandomScaling(scale_range=(0.9, 1.1), p=cfg.prob_augmentations.scaling),
            # RandomChannelRotation(p=cfg.prob_augmentations.channel_rotation),
            # RandomChannelDropout(p=cfg.prob_augmentations.channel_dropout),
            # RandomValueDropout(p=cfg.prob_augmentations.value_dropout)
        ])

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Args:
            x (torch.Tensor): Input tensor to be augmented.
        Returns:
            torch.Tensor: Augmented tensor.

        Method to apply the random data augmentations to the input signal.
        '''
        return self.transform(x)

### Dataset

In [ ]:
class ProjectDataset(Dataset):
    """
    Dataset class for loading ...
    """
    def __init__(self, cfg: Any,
                 split: str,
                 scaler: Optional[Dict[str, np.ndarray]] = None,
                 datapoint_transform: transforms = None,
                 label_transform: transforms = None
        ) -> None:
        """
        Initialize the dataset.

        Args:
            cfg: Configuration object with dataset paths and parameters.
            split: One of 'train', 'val', or 'test'.
            scaler: Optional dictionary {'mean': ..., 'std': ...} for normalization.
        """
        self.data_root = Path(cfg.dataset.path)
        self.cfg = cfg.dataset


        labels_path = self.data_root / 'RawData' / 'labels.txt'
        if not labels_path.exists():
            raise FileNotFoundError(f"Labels file not found at {labels_path}")


        self.samples, self.targets = self._load_data()

        if len(self.samples) == 0:
            raise RuntimeError(
                f"Dataset split '{split}' is empty. "
                f"Check configuration paths or User IDs: {list(self.users)}"
            )


    def _load_data(self) -> Tuple[np.ndarray, np.ndarray]:
        """
        Load raw sensor files, merge them, and segment into datapoints.
        
        Returns:
            Tuple of (samples, labels) where samples is (N, Window, 6).
        """
        all_datapoints = []
        all_labels = []

        # TODO: Implement data loading, merging, and logic here.

        return np.array(all_datapoints, dtype=np.float32), np.array(all_labels, dtype=np.int64)


    def __len__(self) -> int:
        """
        Return the number of valid windows in the dataset.
        """
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Union[int, torch.Tensor]]:
        """
        Return a window and its label.
        
        Output shapes:
        - seq2label: (Window_Size, 6), Scalar Integer
        - seq2seq:   (Window_Size, 6), (Window_Size,) (LongTensor)
        """
        # TODO: Adapt to projects needs
        x = torch.from_numpy(self.samples[idx])
        y = self.targets[idx]

        if self.window_transform:
            x = self.window_transform(x)
        if self.label_transform:
            y = self.label_transform(y)

        return x, y

## Class for general utilities

In [ ]:
class Utilities:
    '''
    Class to encapsulate utilities for training, such as logging and checkpointing.
    '''

    def __init__(self):
        '''
        Class to organize different utility functionalities.
        '''
        pass

    def seed_worker(self, worker_id):
        '''
        Seed worker for DataLoader to ensure reproducibility.
        '''
        _ = worker_id  # Unused, but required by the DataLoader
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    def get_dataset(self, cfg, split='train', scaler=None):
        '''
        Args:
            cfg: Config object
            split: 'train', 'val', or 'test'
            scaler: The scaler dict {'mean':..., 'std':...} from the train set.
                    Only required if split is 'val' or 'test'.
        
        Return:
            dataset
            Dataloader
            current_scaler: scalar stats of the training Data
        '''
        if split == 'train':
            transform_ops = [
                DataAugmentation(cfg.dataset)
            ]
        else:  # no augmentation for test
            transform_ops = [
            ]

        datapoint_transform = T.Compose(transform_ops)
        label_transform = T.Compose([])


        dataset = ProjectDataset(cfg, split, scaler=scaler, window_transform=datapoint_transform,
                        label_transform=label_transform)


        # Generator for reproducibility
        g = torch.Generator()
        g.manual_seed(cfg.seed)

        dataloader = DataLoader(
            dataset, 
            batch_size=cfg.dataset.batch_size, 
            shuffle=(split == 'train'), # Just shuffle during training
            worker_init_fn=self.seed_worker,
            generator=g
        )

        return dataset, dataloader
    
    def save_checkpoint(
        self,
        run_name: str,
        model_state: dict,
        optim_state: dict,
        loss: float,
        epoch: int
    ) -> None:
        '''
        Args:
            run_name (str): Name of the current run
            model_state (dict): State dictionary of the model
            optim_state (dict): State dictionary of the optimizer
            loss (float): Current loss value
            epoch (int): Current epoch number

        Saves the model checkpoint to a file inside the output folder.
        Saving the loss is not necessary, but it might be interesting to look at.
        '''
        # Get path of the current file and build checkpoints dir relative to it
        checkpoint_path = Path(__file__).parents[2]
        checkpoint_path = checkpoint_path / 'outputs'
        # Find newest folder in checkpoints_dir
        for _ in range(2):
            newest_folder = ""
            newest_time = 0
            for folder in checkpoint_path.iterdir():
                if folder.is_dir():
                    dir_time = folder.stat().st_mtime
                    if dir_time > newest_time:
                        newest_time = dir_time
                        newest_folder = folder.name
            checkpoint_path = checkpoint_path / newest_folder
        checkpoint_path = checkpoint_path / f'{run_name}_checkpoint.pth'

        # Create checkpoint dictionary
        checkpoint = {
            'model_state_dict': model_state,
            'optimizer_state_dict': optim_state,
            'loss': loss,
            'epoch': epoch
        }
        torch.save(checkpoint, str(checkpoint_path))
        logging.info('Checkpoint saved at %s', checkpoint_path)


    def load_checkpoint(
        self,
        checkpoint_path: Path,
        model: nn.Module,
        optimizer: torch.optim.Optimizer
    ) -> tuple[nn.Module, torch.optim.Optimizer, float, int]:
        '''
        Args:
            checkpoint_path (str): Path to the checkpoint file
            model (nn.Module): Model to load the state dict into
            optimizer (torch.optim.Optimizer): Optimizer to load the state dict into

        Returns: 
            tuple[nn.Module, torch.optim.Optimizer]: Model and optimizer with loaded state dicts

        Loads a checkpoint from a file and restores the model and optimizer states.
        '''
        checkpoint = torch.load(checkpoint_path, weights_only=True)
        model.load_state_dict(checkpoint['model_state_dict'])
        if optimizer is not None:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint['epoch']
        loss = checkpoint['loss']
        logging.info('Checkpoint loaded from %s', checkpoint_path)
        return model, optimizer, loss, epoch


    def set_seed(self, seed: int):
        '''
        Set random seed for reproducibility across all libraries.

        Args:
            seed (int): Random seed value
        '''
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def get_parameters_table(self, model: torch.nn.Module, position: int = 0) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize
            position (int): The position in the model hierarchy (recursive depth)

        Returns a list of tuples containing (module name, position, number of trainable parameters).
        '''
        # name, position, number trainable parameters
        lines = []
        num_trainable_params = 0
        for _, param in model.named_parameters(recurse=False):
            if param.requires_grad:
                num_trainable_params += math.prod(param.size())

        child_lines = []
        for module in model.children():
            child_lines += self.get_parameters_table(module, position + 1)

        sum_params = 0
        for line in child_lines:
            if line[1] == position + 1:
                sum_params += line[2]

        if num_trainable_params == 0:
            lines.append([type(model).__name__, position, sum_params])
        else:
            lines.append([type(model).__name__, position, num_trainable_params])
        lines += child_lines

        return lines

    def get_model_summary(self, model: torch.nn.Module) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize

        Returns a string representation of the model's summary.
        '''
        lines = self.get_parameters_table(model)
        summary = ""
        for line in lines:
            summary += "  " * line[1] + f"└─ {line[0]}: {line[2]} trainable parameters\n"
        summary += "=================================================================\n"
        summary += f"Total Trainable Parameters: {lines[0][2]}\n"
        return summary

## Trainer class
Class for handling the training of the model

In [ ]:
class Trainer:
    '''
    Class to handle the training loop for a model.
    '''

    def __init__(
        self,
        cfg: any, # Contains the configuration parameters for training, such as learning rate, epochs, etc.
        train_loader: DataLoader,
        model: Module,
        evaluator: Evaluator,
        device: torch.device
    ):
        '''
        Args:
            cfg (object): Configuration object with training parameters.
            train_loader (DataLoader): DataLoader for the training dataset.
            model (nn.Module): The model to be trained.
            evaluator (Evaluator): Evaluator for validation during training.
            device (torch.device): Device to run the training on.

        Initializes the Trainer with the given arguments.
        '''
        self.train_loader = train_loader

        # Ogranize training parameters, so not all are attributes of this class
        self.training_parameters = TrainingParameters(
            lr=cfg.model.lr,
            lr_min=cfg.model.lr_min,
            lr_scheduler=cfg.model.lr_scheduler,
            beta1=cfg.beta1,
            beta2=cfg.beta2,
            weight_decay=cfg.weight_decay,
            epochs=cfg.epochs,
        )
        
        # Create optimizer
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.training_parameters.get_learning_rate(),
            betas=self.training_parameters.get_betas(),
            weight_decay=self.training_parameters.get_weight_decay()
        )

        # Initialize training components
        model.to(device)
        self.model = model
        self.optimizer = optimizer
        self.evaluator = evaluator
        self.device = device

        # Set intervals
        self.log_interval = cfg.log_interval
        self.eval_interval = cfg.eval_interval
        self.checkpoint_interval = cfg.logging.checkpoint_interval

        # Set loss function
        self.criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

        # Change learning rate scheduler based on config
        self.scheduler = None
        if self.training_parameters.lr_scheduler == "cosine":
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.training_parameters.get_epochs(),
                eta_min=self.training_parameters.get_learning_rate_min()
            )



    def get_trained_model(self) -> Module:
        '''
        Returns:
            The trained model

        Function to access the trained model by the Trainer class.
        '''
        return self.model

    def train(self):
        '''
        Main training loop.
        '''

        for epoch in range(self.training_parameters.get_epochs() - self.start_epochs):
            start_time = time.time()
            # Set model to training mode
            self.model.train()
            running_loss = self.start_loss
            correct = 0
            total = 0

            for step, (x, y_true) in enumerate(self.train_loader):
                # TODO: Training loop implementation
                pass

            end_time = time.time()

            if self.scheduler is not None:
                self.scheduler.step()

            if (epoch + 1) % self.eval_interval == 0:
                # Set model to evaluation mode
                self.model.eval()
                with torch.no_grad():
                    self.evaluator.eval()

            if (epoch % self.log_epoch_intervall == 0):
                logging.info(
                    "Epoch [%d/%d]: %s duration", self.start_epochs+epoch+1, self.training_parameters.get_epochs(), end_time - start_time)

            # Save checkpoint if checkpointing is enabled (interval <= total epochs)
            if (self.checkpoint_interval <= self.training_parameters.get_epochs()) and \
               (((epoch + 1) % self.checkpoint_interval == 0) or (epoch+1) == self.training_parameters.get_epochs()):
                    save_checkpoint(
                        run_name="default_run",
                        model_state=self.model.state_dict(),
                        optim_state=self.optimizer.state_dict(),
                        loss=running_loss / len(self.train_loader.dataset),
                        epoch=self.start_epochs + epoch + 1
                    )

        logging.info("Final checkpoint saved")
        logging.info("Training finished")

## Evaluator class
Class for handling the evaluation of the model with the test set, calculating different relevant metrics

In [ ]:
class Evaluator:
    """
    Evaluator for binary and multiclass classification tasks.

    Handles model evaluation with metrics computation for both:
        - Binary classification: num_classes=1 (BCELoss) or num_classes=2 (CrossEntropyLoss)
        - Multiclass classification: num_classes>2 (CrossEntropyLoss)

    Automatically selects metrics (Accuracy, Precision, Recall, F1, AUROC, Confusion Matrix)
    based on the classification task type.
    """

    def __init__(
        self,
        cfg: Any,
        eval_loader: DataLoader,
        model: nn.Module,
        device: torch.device,
        criterion: nn.Module = None
    ):
        """
        Args:
            cfg: Hydra config. Must contain 'num_classes'.
            eval_loader (DataLoader): DataLoader for the evaluation dataset.
            model (nn.Module): The model to evaluate.
            device (torch.device): 'cuda' or 'cpu'.
            criterion (nn.Module, optional): Loss function (nn.BCELoss or nn.CrossEntropyLoss).
                                            Uses 'nn.CrossEntropyLoss' as default.
        """
        self.cfg = cfg
        self.eval_loader = eval_loader
        self.model = model
        self.device = device
        self.mode = cfg.mode

        IGNORE_INDEX = 0

        self.criterion = criterion if criterion is not None else nn.CrossEntropyLoss(
                            ignore_index=IGNORE_INDEX)

        # Raise error if cfg doesnt have 'num_classes' attribute
        if not hasattr(cfg, 'num_classes'):
            raise ValueError(
                "Config object 'cfg' must contain 'num_classes' attribute.")

        self.num_classes = cfg.num_classes
        self.use_bce = isinstance(self.criterion, nn.BCELoss)

        # Define a dict containing "task" and "num_classes" for intializing the metrics
        if self.num_classes <= 2:
            self.task = "binary"
            # No num_classes needed for binary
            self.task_kwargs = {"task": self.task}
        else:  # num_classes > 2
            self.task = "multiclass"
            self.task_kwargs = {"task": self.task,
                                "num_classes": self.num_classes,
                                "ignore_index": IGNORE_INDEX}

        # Creates and fills a dict to store all metrics
        self.metrics = nn.ModuleDict()
        self.metrics.update(self._get_metrics())
        for metric in self.metrics.values():
            metric.to(device)

    def _get_metrics(self) -> dict:
        """
        Build the dictionary of evaluation metrics.

        Binary metrics:
            - Accuracy, Precision, Recall, F1, AUROC, Confusion Matrix

        Multiclass metrics:
            - Accuracy (micro-averaged)
            - Precision, Recall, F1, AUROC (macro-averaged)
            - Confusion Matrix

        Returns:
            dict: Dictionary mapping metric names to torchmetrics instances.
        """

        # Metrics that are common in both binary and multi
        metrics = {
            'conf_matrix': ConfusionMatrix(**self.task_kwargs),
        }

        # Define task specific metrics
        if self.task == "binary":
            metrics.update({
                'accuracy_evaluation': Accuracy(**self.task_kwargs),
                'auroc': AUROC(**self.task_kwargs),
                'precision': Precision(**self.task_kwargs),
                'recall': Recall(**self.task_kwargs),
                'f1': F1Score(**self.task_kwargs),
            })
        else:  # multiclass
            # macro gives each class the same weight, micro each sample
            macro_kwargs = {**self.task_kwargs, "average": "macro"}
            micro_kwargs = {**self.task_kwargs, "average": "micro"}
            metrics.update({
                'accuracy_evaluation': Accuracy(**micro_kwargs),
                'auroc_macro': AUROC(**macro_kwargs),
                'precision_macro': Precision(**macro_kwargs),
                'recall_macro': Recall(**macro_kwargs),
                'f1_macro': F1Score(**macro_kwargs),
            })
        return metrics

    def _reset_metrics(self) -> None:
        """
        Resets all metric states. Called at the beginning of eval.
        """
        for metric in self.metrics.values():
            metric.reset()

    @torch.no_grad()
    def eval(self, return_metrics: bool = False) -> None:
        """
        Args:
            return_metrics: Optionally return metric. Default: False

        Run a full evaluation loop over the evaluation dataset and logs the metrics to
        Weights & Biases.
        """
        self.model.eval()
        self._reset_metrics()

        running_loss = 0.0

        for _, (x, y_true) in enumerate(self.eval_loader):
            x, y_true = x.to(self.device), y_true.to(self.device)

            # TODO: create evaluation loop

        # Compute final metrics for hole dataset after iterating through all batches
        final_metrics = {}

        # Calculate average loss per batch
        final_metrics = {
            'loss_evaluation': running_loss / len(self.eval_loader)}

        # Compute all other metrics
        for metric_name, metric in self.metrics.items():
            try:
                final_metrics[metric_name] = metric.compute()
            except Exception as e:
                logging.warning("[Warning] Could not compute metric '%s'. Set to None. Error: %s",
                                metric_name, e)
                final_metrics[metric_name] = None

        if return_metrics:
            return final_metrics

## Model
Class of the selected model including the definition of the architecture as well as the forward porapagation

In [ ]:
class ModelName:
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        self.name = "ModelName"

        # TODO: Implement model architecture based on cfg.model
        
        pass

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the hybrid model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''

        # TODO: Implement the forward pass of the model

        return x

## Main function

In [ ]:
def main(config=None):
    '''
    Main function to run the training and evaluation pipeline.
    '''
    if config is None:
        raise ValueError("Config dictionary must be provided.")
    
    cfg = CONFIG
    
    # Create utility instance
    util = Utilities()
    
    # Set seed for reproducibility
    util.set_seed(cfg.seed)

    # Select device (GPU if available, else CPU)
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    logging.info('Using device: %s', device)

    # Get Datasets
    _, train_dataloader = util.get_dataset(cfg, split='train')
    _, eval_dataloader, _ = util.get_dataset(cfg, split='val')
    _, test_dataloader, _ = util.get_dataset(cfg, split='test')

    # Get Model
    model = ModelName(cfg.model)
    model_summary = util.get_model_summary(model)
    logging.info(model_summary)


    # Get evaluator
    evaluator = Evaluator(cfg, eval_dataloader, model, device)

    # Get trainer
    trainer = Trainer(cfg,
                      train_dataloader,
                      model,
                      evaluator,
                      device)

    # Train the model and perform final evaluation on test set
    trainer.train()

    test_evaluator = Evaluator(cfg, test_dataloader, trainer.get_trained_model(), device, criterion=None)
    logging.info("Running Evaluation")
    test_evaluator.eval()


if __name__ == '__main__':
    main()
